[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/37_uncertainty_and_calibration/03_aleatoric_and_epistemic.ipynb)

# Bài 03 — Bất định aleatoric và epistemic

**🎯 Mục tiêu**
- Phân biệt hai nguồn bất định: **aleatoric** (nhiễu vốn có của dữ liệu — không giảm được bằng thêm dữ liệu) và **epistemic** (do mô hình thiếu hiểu biết — giảm được khi có thêm dữ liệu).
- Hồi quy **dị phương sai**: mạng dự đoán cả trung bình và phương sai (loss Gaussian NLL) — ước lượng aleatoric.
- **Deep ensemble**: độ phân tán giữa các thành viên đo epistemic; phân rã phương sai toàn phần.
- Phân loại: phân rã **entropy** (tổng = aleatoric + thông tin tương hỗ) và phát hiện dữ liệu **ngoài phân phối** (OOD).
- Giới hạn: chính các ước lượng bất định cũng có thể sai ở vùng chưa thấy.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/37_uncertainty_and_calibration

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

from ml_course.visualization import set_style

try:
    import torch
    from torch import nn
except ImportError as e:
    raise ImportError("Bài này cần PyTorch: pip install -e \".[dl]\"") from e

set_style()
warnings.filterwarnings("ignore")
torch.set_num_threads(1)

## 1. Hai loại bất định

| | Aleatoric | Epistemic |
|---|---|---|
| Nguồn | nhiễu, tính ngẫu nhiên vốn có (đo đạc, hành vi con người, thông tin không quan sát được) | thiếu dữ liệu / thiếu hiểu biết của mô hình |
| Thêm dữ liệu | **không** giảm | giảm |
| Ở đâu lớn | vùng dữ liệu nhiễu | vùng ít/không có dữ liệu, ngoài phân phối |
| Ví dụ | giá nhà cùng đặc điểm vẫn chênh nhau; tung đồng xu | dự đoán giá nhà 500 m² khi chỉ train nhà ≤ 200 m² |

Phân biệt quan trọng cho **hành động**: epistemic cao → thu thêm dữ liệu, hỏi chuyên gia, từ chối dự đoán; aleatoric cao → chấp nhận, báo khoảng dự đoán rộng.

## 2. Dữ liệu 1D có "lỗ hổng" và nhiễu dị phương sai

$y = \sin(1.5x) + 0.3x + \varepsilon$, với $x$ chỉ lấy trong $[-4, -1] \cup [1, 4]$ (không có dữ liệu ở giữa) và độ lệch chuẩn nhiễu **khác nhau**: nhỏ bên trái, lớn dần bên phải.

In [ ]:
rng = np.random.default_rng(0)
n = 600
x = np.r_[rng.uniform(-4, -1, n // 2), rng.uniform(1, 4, n // 2)]
true_sigma = lambda x: 0.15 + 0.25 * np.abs(x) / 4 * (x > 0)
f_true = lambda x: np.sin(1.5 * x) + 0.3 * x
y = f_true(x) + rng.normal(0, 1, n) * true_sigma(x)
X_t, Y_t = torch.tensor(x[:, None], dtype=torch.float32), torch.tensor(y[:, None], dtype=torch.float32)
grid = np.linspace(-6, 6, 241)
G_t = torch.tensor(grid[:, None], dtype=torch.float32)
plt.figure(figsize=(8, 3.5))
plt.scatter(x, y, s=6, alpha=0.5)
plt.plot(grid, f_true(grid), "k--", lw=1, label="hàm thật")
plt.legend()
plt.tight_layout()
plt.show()

## 3. Aleatoric: mạng dự đoán trung bình và phương sai

Mạng xuất hai số: $\mu(x)$ và $\log\sigma^2(x)$; huấn luyện bằng **negative log-likelihood Gaussian**:

$$\mathcal L = \frac12\log\sigma^2(x) + \frac{(y - \mu(x))^2}{2\sigma^2(x)}$$

Số hạng thứ hai phạt sai số lớn, số hạng thứ nhất ngăn mô hình "khai" phương sai lớn tuỳ tiện — mô hình học **mức nhiễu thật** ở từng vùng. Train **5 mạng** với khởi tạo khác nhau (deep ensemble — Lakshminarayanan et al., 2017).

In [ ]:
def train_gaussian_mlp(seed, steps=3000):
    torch.manual_seed(seed)
    net = nn.Sequential(nn.Linear(1, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh(), nn.Linear(64, 2))
    opt = torch.optim.Adam(net.parameters(), lr=3e-3)
    for _ in range(steps):
        out = net(X_t)
        mu, log_var = out[:, :1], out[:, 1:]
        loss = (0.5 * log_var + 0.5 * (Y_t - mu) ** 2 / log_var.exp()).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
    with torch.no_grad():
        out = net(G_t)
    return out[:, 0].numpy(), out[:, 1].exp().numpy()


t0 = time.perf_counter()
members = [train_gaussian_mlp(k) for k in range(5)]
mus, variances = np.array([m for m, _ in members]), np.array([v for _, v in members])
print(f"train 5 mạng: {time.perf_counter() - t0:.0f}s")

## 4. Epistemic: các thành viên ensemble bất đồng ở đâu?

Với ensemble $\{(\mu_k, \sigma_k^2)\}$, phương sai dự đoán toàn phần (luật tổng phương sai):

$$\text{Var}[y\mid x] = \underbrace{\frac1K\sum_k\sigma_k^2(x)}_{\text{aleatoric}} + \underbrace{\frac1K\sum_k\left(\mu_k(x) - \bar\mu(x)\right)^2}_{\text{epistemic}}$$

In [ ]:
aleatoric = variances.mean(0)
epistemic = mus.var(0)
mean = mus.mean(0)
total_std = np.sqrt(aleatoric + epistemic)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(x, y, s=5, alpha=0.3, color="gray")
for m in mus:
    axes[0].plot(grid, m, lw=0.8, alpha=0.7)
axes[0].fill_between(grid, mean - 1.64 * total_std, mean + 1.64 * total_std, alpha=0.2, label="khoảng 90% (toàn phần)")
axes[0].plot(grid, f_true(grid), "k--", lw=1, label="hàm thật")
axes[0].set_ylim(-5, 5)
axes[0].legend(fontsize=8)
axes[0].set_title("5 thành viên ensemble")
axes[1].plot(grid, np.sqrt(aleatoric), label="aleatoric (std)")
axes[1].plot(grid, np.sqrt(epistemic), label="epistemic (std)")
axes[1].plot(grid[np.abs(grid) >= 1], true_sigma(grid[np.abs(grid) >= 1]), "k:", label="σ thật (trong vùng dữ liệu)")
axes[1].set_ylim(0, 1.5)
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

regions = {"dữ liệu trái [−4, −1]": (grid >= -4) & (grid <= -1), "dữ liệu phải [1, 4]": (grid >= 1) & (grid <= 4),
           "lỗ hổng (−0.8, 0.8)": np.abs(grid) < 0.8, "ngoài |x| > 4.5": np.abs(grid) > 4.5}
for name, m in regions.items():
    sig = f" | σ thật TB = {true_sigma(grid[m]).mean():.3f}" if "dữ liệu" in name else ""
    print(f"{name:22s}: aleatoric std = {np.sqrt(aleatoric[m]).mean():.3f} | epistemic std = {np.sqrt(epistemic[m]).mean():.3f}{sig}")

- **Trong vùng có dữ liệu**: aleatoric khớp mức nhiễu thật (nhỏ bên trái, lớn bên phải); epistemic gần 0 — các thành viên đồng thuận.
- **Lỗ hổng và ngoài vùng**: các thành viên **bất đồng** (mỗi mạng ngoại suy một kiểu) → epistemic tăng mạnh — đúng tín hiệu "mô hình không biết".
- Cảnh báo: ngoài vùng dữ liệu, **chính ước lượng aleatoric** cũng là ngoại suy (đầu ra $\log\sigma^2$ tăng/giảm tuỳ ý) — con số aleatoric ở đó không đáng tin. Mọi ước lượng bất định đều là mô hình, và mô hình có thể sai ở nơi chưa thấy.

### Khoảng dự đoán có đúng không?

In [ ]:
x_new = np.r_[rng.uniform(-4, -1, 2000), rng.uniform(1, 4, 2000)]
y_new = f_true(x_new) + rng.normal(0, 1, len(x_new)) * true_sigma(x_new)
for name, std in [("chỉ epistemic", np.sqrt(epistemic)), ("chỉ aleatoric", np.sqrt(aleatoric)), ("toàn phần", total_std)]:
    lo, hi = np.interp(x_new, grid, mean - 1.64 * std), np.interp(x_new, grid, mean + 1.64 * std)
    print(f"khoảng 90% dùng {name:14s}: tỉ lệ phủ thực tế = {np.mean((y_new >= lo) & (y_new <= hi)):.1%}")

Trong vùng có dữ liệu, khoảng chỉ dựa vào epistemic quá hẹp (bỏ qua nhiễu) — phủ chưa tới 5%; aleatoric chiếm gần như toàn bộ bất định ở đây. Khoảng toàn phần phủ gần 90% nhưng **thấp hơn** một chút: ước lượng phương sai của mạng chưa hoàn hảo, và khoảng Gaussian chỉ đúng nếu mô hình đúng. Không có gì **đảm bảo** độ phủ — bài 04 (conformal) giải quyết đúng vấn đề này mà không cần giả định Gaussian.

## 5. Phân loại: phân rã entropy và phát hiện OOD

Với ensemble phân loại, độ bất định toàn phần là entropy của xác suất trung bình; tách thành:

$$\underbrace{H\Big[\tfrac1K\textstyle\sum_k p_k\Big]}_{\text{toàn phần}} = \underbrace{\tfrac1K\textstyle\sum_k H[p_k]}_{\text{aleatoric (từng mô hình đều không chắc)}} + \underbrace{\text{MI}}_{\text{epistemic (các mô hình bất đồng)}}$$

Ensemble 5 MLP train trên chữ số 8×8, thử với dữ liệu test thật và ba loại dữ liệu **ngoài phân phối**: nhiễu ngẫu nhiên, chữ số **đảo màu**, và ảnh gần như trống.

In [ ]:
Xd, yd = load_digits(return_X_y=True)
Xd = Xd / 16
d_train, d_test = train_test_split(np.arange(len(Xd)), test_size=0.3, random_state=0, stratify=yd)
t0 = time.perf_counter()
ensemble = [MLPClassifier((128,), alpha=1e-3, max_iter=1000, random_state=k).fit(Xd[d_train], yd[d_train]) for k in range(5)]
print(f"train ensemble: {time.perf_counter() - t0:.0f}s | accuracy test = {np.mean(np.mean([m.predict_proba(Xd[d_test]) for m in ensemble], 0).argmax(1) == yd[d_test]):.3f}")

r = np.random.default_rng(0)
datasets = {"chữ số (test)": Xd[d_test], "nhiễu ngẫu nhiên": r.random((len(d_test), 64)),
            "chữ số đảo màu": 1 - Xd[d_test], "ảnh gần như trống": np.clip(r.normal(0.1, 0.1, (len(d_test), 64)), 0, 1)}
entropy = lambda P: -(P * np.log(P + 1e-12)).sum(-1)
scores = {}
for name, data in datasets.items():
    P = np.array([m.predict_proba(data) for m in ensemble])
    P_mean = P.mean(0)
    total, alea = entropy(P_mean), entropy(P).mean(0)
    scores[name] = {"toàn phần": total, "aleatoric": alea, "epistemic (MI)": total - alea, "1 − max prob": 1 - P_mean.max(1)}
    print(f"{name:18s}: max prob TB = {P_mean.max(1).mean():.3f} | entropy toàn phần = {total.mean():.3f} | "
          f"aleatoric = {alea.mean():.3f} | epistemic = {(total - alea).mean():.3f}")

print("\nROC-AUC phân biệt OOD với chữ số thật theo từng điểm số:")
for name in list(datasets)[1:]:
    labels = np.r_[np.zeros(len(d_test)), np.ones(len(d_test))]
    print(f"  {name:18s}: " + " | ".join(f"{k} = {roc_auc_score(labels, np.r_[scores['chữ số (test)'][k], scores[name][k]]):.3f}" for k in scores[name]))

- Mô hình vẫn đưa xác suất lớn nhất **cao** cho nhiễu và ảnh đảo màu — mạng phân loại không "biết mình không biết" chỉ qua softmax.
- Với nhiễu và ảnh đảo màu, **epistemic (MI)** — các thành viên bất đồng — phát hiện OOD tốt hơn độ tự tin.
- Với ảnh gần như trống, mọi mô hình đều thấy "lớp nào cũng có thể" → entropy **aleatoric** cao mới là tín hiệu tốt nhất.
- Không có điểm số OOD nào tốt cho mọi loại dữ liệu lạ; phát hiện OOD đáng tin cần kiểm tra trên **đúng loại** dữ liệu lạ có thể gặp khi triển khai.

## 6. Các cách ước lượng epistemic khác

- **Bayesian**: phân phối hậu nghiệm trên tham số (chương 23: hồi quy Bayes, Gaussian process — epistemic tự nhiên tăng xa dữ liệu).
- **MC dropout** (Gal & Ghahramani, 2016): giữ dropout khi suy luận, lặp nhiều lần — xấp xỉ rẻ của ensemble.
- **Bootstrap** cho mô hình cổ điển; **Laplace approximation**; **mô hình nhiều đầu** (multi-head).
- Deep ensemble thường là baseline mạnh nhất dù đơn giản (Ovadia et al., 2019), đổi lại tốn K lần chi phí.

Ứng dụng: **active learning** (gán nhãn nơi epistemic cao), **từ chối dự đoán** (chuyển cho người), phát hiện **dữ liệu trôi** (chương 40), khám phá trong RL (UCB — chương 35).

## 🧠 Kiểm tra nhanh

1. Phân biệt aleatoric và epistemic. Loại nào giảm khi thêm dữ liệu?
2. Viết loss Gaussian NLL. Vì sao nó học được phương sai?
3. Deep ensemble ước lượng epistemic thế nào? Phân rã phương sai toàn phần?
4. Vì sao ước lượng aleatoric ngoài vùng dữ liệu không đáng tin?
5. Phân rã entropy cho phân loại gồm những phần nào? Phần nào phát hiện OOD tốt hơn trong thí nghiệm?
6. Vì sao softmax max prob không đủ để phát hiện dữ liệu lạ?

## 📌 Tóm tắt

- Aleatoric = nhiễu vốn có (học bằng mô hình dự đoán phân phối, vd Gaussian NLL); epistemic = thiếu hiểu biết (đo bằng sự bất đồng của ensemble/hậu nghiệm).
- Phương sai toàn phần = TB aleatoric + phương sai giữa các thành viên; entropy toàn phần = aleatoric + MI.
- Epistemic tăng ở vùng thiếu dữ liệu và OOD; nhưng mọi ước lượng bất định đều có thể sai ở vùng chưa thấy — kiểm tra trên đúng loại tình huống.

## 📚 Tài liệu tham khảo

- Kendall & Gal (2017) — *What Uncertainties Do We Need in Bayesian Deep Learning for Computer Vision?*
- Lakshminarayanan, Pritzel & Blundell (2017) — *Simple and Scalable Predictive Uncertainty Estimation using Deep Ensembles*
- Gal & Ghahramani (2016) — *Dropout as a Bayesian Approximation*
- Ovadia et al. (2019) — *Can You Trust Your Model's Uncertainty? Evaluating Predictive Uncertainty Under Dataset Shift*
- Hüllermeier & Waegeman (2021) — *Aleatoric and epistemic uncertainty in machine learning: an introduction to concepts and methods*